# **Extraction and Correction of Argentine National Identity Documents with YOLO OBB**

## **Objective**
To extract and correct the perspective of Argentine National Identity Documents (DNI) from the input images using the inference of the **YOLO26s OBB** model adjusted in the previous stage.

## **Technical Justification**
The extracted images are persisted at a fixed resolution of **1200 x 756 pixels** to maintain the actual aspect ratio established by the international standard **ISO/IEC 7810 (ID-1)** ($85.60\text{ mm} \times 53.98\text{ mm} \approx 1.5858$). 

Maintaining the exact aspect ratio prevents mechanical distortion in the internal regions of interest (PDF417, MRZ or transaction numbers) during the application of the homography (`cv2.warpPerspective`).

## **Expected Input**
* **Input Directory:** `data/prc/DNIs/input`.
* **Volume:** ~314 images containing ~324 instances of identity documents.

## **Expected output**
* **Front:** `data/prc/DNIs/processed/front/`.
* **Back:** `data/prc/DNIs/processed/back/`.
* Individually rectified `.png` files at 1200 x 756 px for each detected ID card.

## **Example**
<table>
  <tr>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_1.png" width="420"><br>
      <em>Figure 1. Input image.</em>
    </td>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_2.png" width="420"><br>
      <em>Figura 2. Extracted ID-Card.</em>
    </td>
  </tr>
</table>

## **Libraries and Constants**


In [7]:
from __future__ import annotations
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import ultralytics
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

if not os.environ.get("BASE_PROJECT_DIR"):
    os.environ["BASE_PROJECT_DIR"] = input("Enter the base project directory path: ")

BASE_INPUT_DIR = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/input"
OUTPUT_IMAGES_DIR = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed"
OUTPUT_FRONT_IMAGES = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed/front"
OUTPUT_BACK_IMAGES = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed/back"
YOLO_OBB_MODEL_PATH = Path(os.environ["BASE_PROJECT_DIR"]) / "weights/yolo/yolo26s-obb-DNI-det.pt"
IMAGES = list(BASE_INPUT_DIR.glob("*"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_WIDTH = 1200
TARGET_HEIGHT = 756

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Device: {DEVICE}\n\n")

# Check if the output directory exists, if not, create it
for directory in [OUTPUT_IMAGES_DIR, OUTPUT_FRONT_IMAGES, OUTPUT_BACK_IMAGES]:
    if os.path.exists(directory):
        print(f"Output directory '{directory}' already exists.")
    else:
        os.makedirs(directory)
        print(f"Output directory '{directory}' created.")
        
model = YOLO(YOLO_OBB_MODEL_PATH)  # load a pretrained model (recommended for training)

Ultralytics version: 8.4.117
Numpy version: 2.3.5
OpenCV version: 4.10.0
PyTorch version: 2.13.0+cu130
Matplotlib version: 3.11.0
Device: cuda


Output directory '/home/nahuel/Documentos/Python/DNI/data/prc/DNIs/processed' created.
Output directory '/home/nahuel/Documentos/Python/DNI/data/prc/DNIs/processed/front' created.
Output directory '/home/nahuel/Documentos/Python/DNI/data/prc/DNIs/processed/back' created.


In [ ]:
def order_points(points):
    points = np.array(points, dtype=np.float32)

    s = points.sum(axis=1)
    diff = np.diff(points, axis=1).flatten()

    top_left = points[np.argmin(s)]
    top_right = points[np.argmin(diff)]
    bottom_right = points[np.argmax(s)]
    bottom_left = points[np.argmax(diff)]

    return np.array([
        top_left,
        top_right,
        bottom_right,
        bottom_left
    ], dtype=np.float32)


def rectify_obb(image, obb_points, target_size=(1200, 756)):
    src = order_points(obb_points)

    target_width, target_height = target_size

    # Width and height of the detected document
    width = np.linalg.norm(src[1] - src[0])
    height = np.linalg.norm(src[3] - src[0])

    # --------------------------------------------------
    # If the document is vertical, rotate the points to make it horizontal
    # --------------------------------------------------
    if height > width:
        # Rotate the order of the points
        src = np.array([
            src[3],  # top-left
            src[0],  # top-right
            src[1],  # bottom-right
            src[2]   # bottom-left
        ], dtype=np.float32)

    dst = np.array([
        [0, 0],
        [target_width - 1, 0],
        [target_width - 1, target_height - 1],
        [0, target_height - 1]
    ], dtype=np.float32)

    # Homography
    matrix = cv2.getPerspectiveTransform(src, dst)

    # Transformation
    warped = cv2.warpPerspective(
        image,
        matrix,
        (target_width, target_height)
    )

    return warped

def extract_dni_from_image(image_path: Path,
                           output_front: Path,
                           output_back: Path,
                           model: YOLO,
                           device: str) -> True:
    try:
        result = model.predict(source=image_path,
                            device=device,
                            verbose=False)[0]

        if len(result.obb) == 0:
            return False

        for i, obb in enumerate(result.obb):
            img = cv2.imread(str(image_path))
            idx = obb.conf.argmax()
            cls = obb.cls[idx].cpu().numpy().astype(np.int32)
            points = obb.xyxyxyxy[idx].cpu().numpy().astype(np.int32)
            dni = rectify_obb(img, points, target_size=(TARGET_WIDTH, TARGET_HEIGHT))
            if cls == 0:
                cv2.imwrite(output_front / f"{image_path.stem}_{i}.png", dni)
            else:
                cv2.imwrite(output_back / f"{image_path.stem}_{i}.png", dni)
        return True
    except Exception as e:
        return False

## **Execute the pipeline**

In [9]:
error_count = 0
for image in tqdm(IMAGES, desc="Processing images"):
    success = extract_dni_from_image(image,
                           OUTPUT_FRONT_IMAGES,
                           OUTPUT_BACK_IMAGES,
                           model,
                           DEVICE)
    if not success:
        error_count += 1
print(f"Finished processing images with {error_count} errors.")

Processing images: 100%|██████████| 314/314 [00:22<00:00, 14.01it/s]

Finished processing images with 1 errors.


In [14]:
fronts = list(OUTPUT_FRONT_IMAGES.glob("*"))
backs = list(OUTPUT_BACK_IMAGES.glob("*"))

print(f"Extracted {len(fronts)} front instances and {len(backs)} back instances of Argentine ID Card.")
print(f"Total {len(fronts) + len(backs)} instances extracted from {len(IMAGES)} images.")

Extracted 188 front instances and 134 back instances of Argentine ID Card.
Total 322 instances extracted from 314 images.


## **Conclusions**

* **Detection Effectiveness:** Our *fine-tuned* **YOLO26s OBB** detected correctly 322 of 324 instances expected in our 314 images, achieving a success rate of 99.4%.
* **Classify by class:** it was extracted and classify 188 fronts and 134 backs of DNIs.
* **Geometric Normalization:** was successfully applied the homography transform (`cv2.warpPerspective`), correcting the perspective of each ID-Card to a size of 1200 x 756 px. To this way preserv the real aspect ratio and maintain the ISO/IEC 7810 (ID-1) standard without distortions in critics regions as PDF417 or MRZ.
* **Integrity:** this notebook no only measure if the DNI was detected, but it confirms the correct separation between front and back.